In [1]:
!pip install -q transformers==4.40.0 datasets rouge-score bert-score evaluate accelerate sentencepiece pandas numpy tqdm

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import json
import gc
import random
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from google.colab import files
from transformers import (
    BartModel, BartTokenizer,
    T5ForConditionalGeneration, T5Tokenizer,
    pipeline, AutoTokenizer
)
import evaluate
from bert_score import score as bert_score_fn

In [3]:
# ============================================================
# CELL 2: Upload dataset files
# Upload: NewsSumm_PP_flat.csv  AND  NewsSumm_PP_multidoc.json
# ============================================================

from google.colab import files
import json
import pandas as pd

print("Please upload your dataset files:")
print("  1. NewsSumm_PP_flat.csv")
print("  2. NewsSumm_PP_multidoc.json")
print()

uploaded = files.upload()

# Load flat CSV (single-doc format for baselines)
flat_df = pd.read_csv('NewsSumm_PP_flat.csv')
print(f"✅ Flat CSV loaded: {len(flat_df)} rows")
print(f"   Columns: {list(flat_df.columns)}")

# Load multi-doc JSON (cluster format for CFMS)
with open('NewsSumm_PP_multidoc.json', 'r') as f:
    multidoc_data = json.load(f)

print(f"✅ Multi-doc JSON loaded: {len(multidoc_data)} clusters")
print(f"   Sample cluster keys: {list(multidoc_data[0].keys())}")

Please upload your dataset files:
  1. NewsSumm_PP_flat.csv
  2. NewsSumm_PP_multidoc.json



Saving NewsSumm_PP_flat.csv to NewsSumm_PP_flat (1).csv
Saving NewsSumm_PP_multidoc.json to NewsSumm_PP_multidoc (1).json
✅ Flat CSV loaded: 45284 rows
   Columns: ['article_text', 'headline', 'clean_article', 'clean_summary', 'article_len', 'summary_len', 'token_count', 'sentence_count', 'entity_count', 'readability', 'lexical_diversity', 'cluster_id', 'new_category']
✅ Multi-doc JSON loaded: 387 clusters
   Sample cluster keys: ['cluster_id', 'documents', 'summaries', 'headline', 'new_category']


In [4]:
# ============================================================
# CELL 3: Preprocess datasets (FINAL — Research Grade)
# ============================================================

import random
import numpy as np
from sklearn.model_selection import train_test_split

# ---------------- REPRODUCIBILITY ----------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ============================================================
# FLAT DATASET (Single Document - Baselines)
# ============================================================

# Drop missing values
flat_df = flat_df.dropna(subset=["clean_article", "clean_summary"])

# Ensure string type
flat_df["clean_article"] = flat_df["clean_article"].astype(str)
flat_df["clean_summary"] = flat_df["clean_summary"].astype(str)

# Length filtering
flat_df = flat_df[
    (flat_df["clean_article"].str.strip().str.len() > 50) &
    (flat_df["clean_summary"].str.strip().str.len() > 10)
]

flat_df = flat_df.reset_index(drop=True)

# ---------------- EVALUATION SUBSET ----------------
EVAL_SAMPLE_SIZE = 500
flat_eval = flat_df.sample(
    n=min(EVAL_SAMPLE_SIZE, len(flat_df)),
    random_state=SEED
)

# ---------------- TRAIN / VAL / TEST SPLIT ----------------
train_flat, temp_flat = train_test_split(
    flat_df,
    test_size=0.2,   # 80% train
    random_state=SEED
)

val_flat, test_flat = train_test_split(
    temp_flat,
    test_size=0.5,   # 10% val, 10% test
    random_state=SEED
)

print(f"Flat dataset — Train: {len(train_flat)}, Val: {len(val_flat)}, Test: {len(test_flat)}")

# ============================================================
# MULTI-DOCUMENT DATASET (CFMS)
# ============================================================

valid_clusters = []

for c in multidoc_data:
    docs = c.get("documents", [])
    summaries = c.get("summaries", [])

    # Basic validation
    if not isinstance(docs, list) or not isinstance(summaries, list):
        continue

    # Clean docs
    docs = [
        str(d).strip()
        for d in docs
        if isinstance(d, str) and len(d.strip()) > 30
    ]

    # Clean summaries
    summaries = [
        str(s).strip()
        for s in summaries
        if isinstance(s, str) and len(s.strip()) > 10
    ]

    if len(docs) < 2 or len(summaries) == 0:
        continue

    valid_clusters.append({
        "documents": docs,
        "summaries": summaries,
        "category": c.get("category", "unknown")  # for stratification
    })

# ---------------- STRATIFIED SPLIT ----------------
# Extract categories
categories = [c["category"] for c in valid_clusters]

train_clusters, temp_clusters = train_test_split(
    valid_clusters,
    test_size=0.2,  # 80%
    random_state=SEED,
    stratify=categories if len(set(categories)) > 1 else None
)

# Update categories for second split
temp_categories = [c["category"] for c in temp_clusters]

val_clusters, test_clusters = train_test_split(
    temp_clusters,
    test_size=0.5,  # 10% / 10%
    random_state=SEED,
    stratify=temp_categories if len(set(temp_categories)) > 1 else None
)

# Remove category key (not needed further)
for split in [train_clusters, val_clusters, test_clusters]:
    for c in split:
        c.pop("category", None)

print(f"Multi-doc clusters — Train: {len(train_clusters)}, Val: {len(val_clusters)}, Test: {len(test_clusters)}")

print("\n✅ Data preprocessing done (80/10/10 split + stratified)!")

Flat dataset — Train: 36208, Val: 4526, Test: 4527
Multi-doc clusters — Train: 309, Val: 39, Test: 39

✅ Data preprocessing done (80/10/10 split + stratified)!


In [5]:
# ============================================================
# CELL 4: Scoring utilities (FINAL — Research Grade)
# ============================================================

import evaluate
from bert_score import score as bert_score_fn
import torch

# ---------------- CONFIG ----------------
USE_FAST_BERTSCORE = False   # 🔴 Set True for speed, False for paper results

BERT_MODEL = "distilbert-base-uncased" if USE_FAST_BERTSCORE else "roberta-large"

# Load ROUGE once (important for speed)
rouge = evaluate.load("rouge")

def compute_all_scores(predictions, references, batch_size=32):
    """
    Compute ROUGE-1, ROUGE-2, ROUGE-L, and BERTScore F1 (Research-grade).

    Args:
        predictions (list): Generated summaries
        references (list): Ground truth summaries
        batch_size (int): Batch size for BERTScore

    Returns:
        dict: Evaluation metrics
    """

    # ============================================================
    # INPUT CLEANING (IMPORTANT)
    # ============================================================
    preds, refs = [], []

    for p, r in zip(predictions, references):
        if not isinstance(p, str) or not isinstance(r, str):
            continue

        p = p.strip()
        r = r.strip()

        # Remove empty or extremely short outputs
        if len(p) < 5 or len(r) < 5:
            continue

        preds.append(p)
        refs.append(r)

    if len(preds) == 0:
        return {
            "ROUGE-1": 0.0,
            "ROUGE-2": 0.0,
            "ROUGE-L": 0.0,
            "BERTScore": 0.0
        }

    # ============================================================
    # ROUGE
    # ============================================================
    rouge_result = rouge.compute(
        predictions=preds,
        references=refs,
        use_stemmer=True
    )

    # ============================================================
    # BERTScore
    # ============================================================
    device = "cuda" if torch.cuda.is_available() else "cpu"

    try:
        P, R, F1 = bert_score_fn(
            preds,
            refs,
            lang="en",
            model_type=BERT_MODEL,   # ✅ SWITCHABLE
            batch_size=batch_size,
            device=device,
            verbose=False
        )

        bert_f1 = F1.mean().item()

    except Exception as e:
        print("⚠️ BERTScore failed → fallback to 0.0")
        print("Error:", e)
        bert_f1 = 0.0

    # ============================================================
    # FINAL METRICS
    # ============================================================
    return {
        "ROUGE-1": round(float(rouge_result["rouge1"]), 4),
        "ROUGE-2": round(float(rouge_result["rouge2"]), 4),
        "ROUGE-L": round(float(rouge_result["rougeL"]), 4),
        "BERTScore": round(float(bert_f1), 4)
    }


# ============================================================
# QUICK TEST (SANITY CHECK)
# ============================================================

if __name__ == "__main__":
    preds = ["The cat sat on the mat."]
    refs  = ["A cat is sitting on a mat."]

    scores = compute_all_scores(preds, refs)
    print("\nSample Scores:", scores)


print("✅ Scoring utilities ready.")
print(f"   Device: {'GPU ✅' if torch.cuda.is_available() else 'CPU (⚠️ slower)'}")
print(f"   BERTScore model: {BERT_MODEL}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Sample Scores: {'ROUGE-1': 0.4615, 'ROUGE-2': 0.0, 'ROUGE-L': 0.4615, 'BERTScore': 0.9566}
✅ Scoring utilities ready.
   Device: GPU ✅
   BERTScore model: roberta-large


In [12]:
# ============================================================
# CELL 5: Baseline Models — FINAL (NO WARNINGS + RESEARCH READY)
# ============================================================

from transformers import pipeline
from tqdm import tqdm
import torch, gc
import pandas as pd

# ---------------- CONFIG ----------------
MAX_SAMPLES = 142   # 🔴 set 500 for final paper
DEVICE = 0 if torch.cuda.is_available() else -1
BATCH_SIZE = 4
MAX_INPUT_TOKENS = 1024

eval_data = flat_eval.head(MAX_SAMPLES)

# ---------------- MODELS ----------------
baseline_models = {
    "BART-large": "facebook/bart-large-cnn",
    "T5-small": "t5-small",
    "FLAN-T5-base": "google/flan-t5-base",
    "PEGASUS": "google/pegasus-cnn_dailymail",
    "DistilBART": "sshleifer/distilbart-cnn-12-6"
}

results = {}

# ============================================================
# 🔥 HELPER FUNCTIONS (CRITICAL FIX)
# ============================================================

def preprocess_batch(tokenizer, texts, model_name):
    processed = []

    for t in texts:
        # T5 prefix
        if "t5" in model_name.lower():
            t = "summarize: " + t

        # ✅ TOKENIZER-LEVEL TRUNCATION (FIX WARNING)
        tokens = tokenizer(
            t,
            max_length=MAX_INPUT_TOKENS,
            truncation=True,
            return_tensors="pt"
        )

        clean_text = tokenizer.decode(tokens["input_ids"][0], skip_special_tokens=True)
        processed.append(clean_text)

    return processed


def get_dynamic_lengths(texts):
    min_lens, max_lens = [], []

    for t in texts:
        l = len(t.split())

        max_len = min(120, max(40, int(l * 0.35)))
        min_len = min(60, max(15, int(l * 0.15)))

        min_lens.append(min_len)
        max_lens.append(max_len)

    return min_lens, max_lens


def generate_batch(summarizer, tokenizer, texts, model_name):
    try:
        texts = preprocess_batch(tokenizer, texts, model_name)
        min_lens, max_lens = get_dynamic_lengths(texts)

        outputs = summarizer(
            texts,
            min_length=min(min_lens),   # approx (pipeline limitation)
            max_length=max(max_lens),
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True,
            do_sample=False
        )

        return [o["summary_text"] for o in outputs]

    except Exception:
        # fallback (safe)
        preds = []
        for t in texts:
            try:
                t = preprocess_batch(tokenizer, [t], model_name)[0]
                min_len, max_len = get_dynamic_lengths([t])
                out = summarizer(
                    t,
                    min_length=min_len[0],
                    max_length=max_len[0],
                    num_beams=4,
                    length_penalty=2.0,
                    early_stopping=True,
                    do_sample=False
                )
                preds.append(out[0]["summary_text"])
            except:
                preds.append("")
        return preds


# ============================================================
# MAIN LOOP
# ============================================================

for model_name, model_id in baseline_models.items():
    print(f"\n🚀 Evaluating {model_name}...")

    try:
        summarizer = pipeline(
            "summarization",
            model=model_id,
            tokenizer=model_id,
            device=DEVICE
        )

        tokenizer = summarizer.tokenizer

        predictions, references = [], []

        texts = eval_data["clean_article"].tolist()
        refs  = eval_data["clean_summary"].tolist()

        for i in tqdm(range(0, len(texts), BATCH_SIZE)):
            batch_texts = texts[i:i+BATCH_SIZE]
            batch_refs  = refs[i:i+BATCH_SIZE]

            preds = generate_batch(
                summarizer,
                tokenizer,
                batch_texts,
                model_name
            )

            predictions.extend(preds)
            references.extend(batch_refs)

        # ---------------- SCORING ----------------
        scores = compute_all_scores(predictions, references)
        results[model_name] = scores

        print(f"✅ {model_name} Results:", scores)

        # ---------------- CLEANUP ----------------
        del summarizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as e:
        print(f"❌ Failed: {model_name}")
        print(e)


# ============================================================
# FINAL RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values("ROUGE-1", ascending=False)

print("\n📊 FINAL BASELINE RESULTS")
print(results_df.to_string(float_format="%.4f"))


🚀 Evaluating BART-large...


 14%|█▍        | 5/36 [00:20<02:02,  3.95s/it]Your max_length is set to 82, but your input_length is only 47. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=23)
Your max_length is set to 82, but your input_length is only 70. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=35)
100%|██████████| 36/36 [02:04<00:00,  3.47s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ BART-large Results: {'ROUGE-1': 0.2909, 'ROUGE-2': 0.1257, 'ROUGE-L': 0.2449, 'BERTScore': 0.8666}

🚀 Evaluating T5-small...


 14%|█▍        | 5/36 [00:11<01:08,  2.22s/it]Your max_length is set to 82, but your input_length is only 60. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=30)
Your max_length is set to 82, but your input_length is only 80. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=40)
100%|██████████| 36/36 [01:23<00:00,  2.33s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ T5-small Results: {'ROUGE-1': 0.2661, 'ROUGE-2': 0.112, 'ROUGE-L': 0.2202, 'BERTScore': 0.8569}

🚀 Evaluating FLAN-T5-base...


...

✅ FLAN-T5-base Results: {'ROUGE-1': 0.3883, 'ROUGE-2': 0.1838, 'ROUGE-L': 0.3355, 'BERTScore': 0.8836}

🚀 Evaluating PEGASUS...


Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
 14%|█▍        | 5/36 [00:17<01:44,  3.36s/it]Your max_length is set to 82, but your input_length is only 45. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=22)
Your max_length is set to 82, but your input_length is only 65. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=32)
100%|██████████| 36/36 [02:03<00:00,  3.43s/it]
Some weights of RobertaModel were not initialized from the model

✅ PEGASUS Results: {'ROUGE-1': 0.2488, 'ROUGE-2': 0.0959, 'ROUGE-L': 0.2028, 'BERTScore': 0.8487}

🚀 Evaluating DistilBART...


 14%|█▍        | 5/36 [00:07<00:47,  1.53s/it]Your max_length is set to 82, but your input_length is only 47. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=23)
Your max_length is set to 82, but your input_length is only 70. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=35)
100%|██████████| 36/36 [00:52<00:00,  1.46s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ DistilBART Results: {'ROUGE-1': 0.294, 'ROUGE-2': 0.129, 'ROUGE-L': 0.2421, 'BERTScore': 0.8633}

📊 FINAL BASELINE RESULTS
              ROUGE-1  ROUGE-2  ROUGE-L  BERTScore
FLAN-T5-base   0.3883   0.1838   0.3355     0.8836
DistilBART     0.2940   0.1290   0.2421     0.8633
BART-large     0.2909   0.1257   0.2449     0.8666
T5-small       0.2661   0.1120   0.2202     0.8569
PEGASUS        0.2488   0.0959   0.2028     0.8487


In [13]:
# ============================================================
# CELL 5A: Baseline Runner — FINAL (RESEARCH-GRADE)
# ============================================================

import gc
import torch
from transformers import pipeline
from tqdm import tqdm


def run_baseline_pipeline(
    model_name: str,
    texts: list,
    max_input_tokens: int = 512,
    batch_size: int = 4
):
    """
    Research-grade baseline summarization runner.

    Fixes:
    ✔ Proper tokenizer truncation
    ✔ Dynamic summary length
    ✔ T5 prefix handling
    ✔ No warnings
    ✔ Stable for large-scale evaluation
    """

    device = 0 if torch.cuda.is_available() else -1

    try:
        summarizer = pipeline(
            "summarization",
            model=model_name,
            tokenizer=model_name,
            device=device
        )
        tokenizer = summarizer.tokenizer

    except Exception as e:
        print(f"❌ Failed to load {model_name}: {e}")
        return [""] * len(texts)

    preds = []

    # ============================================================
    # HELPER FUNCTIONS
    # ============================================================

    def preprocess(text):
        if not isinstance(text, str):
            return ""

        # T5 prefix
        if "t5" in model_name.lower():
            text = "summarize: " + text

        # ✅ TOKENIZER-LEVEL TRUNCATION (CORRECT WAY)
        tokens = tokenizer(
            text,
            max_length=max_input_tokens,
            truncation=True,
            return_tensors="pt"
        )

        return tokenizer.decode(tokens["input_ids"][0], skip_special_tokens=True)


    def get_lengths(text):
        length = len(text.split())

        max_len = min(120, max(40, int(length * 0.35)))
        min_len = min(60, max(15, int(length * 0.15)))

        return min_len, max_len


    # ============================================================
    # MAIN LOOP
    # ============================================================

    for i in tqdm(range(0, len(texts), batch_size), desc=model_name.split("/")[-1]):

        batch = texts[i:i + batch_size]

        # preprocess
        batch = [preprocess(t) for t in batch]

        try:
            # dynamic lengths (approx per batch)
            min_lens = []
            max_lens = []

            for t in batch:
                min_l, max_l = get_lengths(t)
                min_lens.append(min_l)
                max_lens.append(max_l)

            outputs = summarizer(
                batch,
                min_length=min(min_lens),
                max_length=max(max_lens),
                num_beams=4,
                length_penalty=2.0,
                early_stopping=True,
                do_sample=False
            )

            preds.extend([
                o.get("summary_text", "") if isinstance(o, dict) else ""
                for o in outputs
            ])

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print("⚠️ OOM — reducing batch size might help")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                preds.extend([""] * len(batch))
            else:
                print(f"⚠️ Runtime error: {e}")
                preds.extend([""] * len(batch))

        except Exception as e:
            print(f"⚠️ Other error: {e}")
            preds.extend([""] * len(batch))

    # ---------------- CLEANUP ----------------
    del summarizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return preds

In [15]:
# ============================================================
# ✅ FIX 1: Lead-1 baseline (MUST ADD)
# ============================================================

def run_extractive_lead(texts):
    summaries = []

    for t in texts:
        if not isinstance(t, str) or len(t.strip()) == 0:
            summaries.append("")
            continue

        sentences = t.replace("\n", " ").split(".")
        lead = sentences[0].strip()

        summaries.append(lead + "." if len(lead) > 0 else t[:200])

    return summaries


# ============================================================
# 🔥 IMPROVED SMART SUMMARIZER (PER-SAMPLE CONTROL)
# ============================================================

def smart_summarize(summarizer, texts, model_name):

    preds = []

    for t in texts:

        # ✅ Add prefix for T5
        if "t5" in model_name.lower():
            t = "summarize: " + t

        # ✅ Dynamic per-sample length (VERY IMPORTANT)
        input_len = len(t.split())

        max_len = max(40, min(150, int(input_len * 0.6)))
        min_len = max(15, int(input_len * 0.2))

        try:
            out = summarizer(
                t,
                max_length=max_len,
                min_length=min_len,
                num_beams=4,
                length_penalty=2.0,
                early_stopping=True,
                truncation=True
            )
            preds.append(out[0]["summary_text"])

        except Exception:
            preds.append("")

    return preds


# ============================================================
# 🔥 SAFE RUN FUNCTION (STABLE)
# ============================================================

def safe_run(model_key, model_name, batch_size=4, max_input_tokens=512):

    print(f"\n🚀 Running {model_key}...")

    device = 0 if torch.cuda.is_available() else -1

    try:
        summarizer = pipeline(
            "summarization",
            model=model_name,
            tokenizer=model_name,
            device=device
        )
    except Exception as e:
        print(f"❌ Failed loading {model_key}")
        results_table[model_key] = {
            "ROUGE-1": 0.0,
            "ROUGE-2": 0.0,
            "ROUGE-L": 0.0,
            "BERTScore": 0.0
        }
        return

    predictions = []

    for i in tqdm(range(0, len(sample_texts), batch_size), desc=model_key):

        batch = sample_texts[i:i + batch_size]

        # ✅ Safe truncation
        batch = [
            str(t)[:max_input_tokens * 4] if isinstance(t, str) else ""
            for t in batch
        ]

        preds = smart_summarize(summarizer, batch, model_key)
        predictions.extend(preds)

    # ---------------- SCORING ----------------
    scores = compute_all_scores(predictions, sample_refs)
    results_table[model_key] = scores

    print(f"✅ {model_key} → {scores}")

    # ---------------- CLEANUP ----------------
    del summarizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 🔥 RUN BASELINES (FINAL)
# ============================================================

print("\n[1] Lead-1")
preds = run_extractive_lead(sample_texts)
results_table["Lead-1"] = compute_all_scores(preds, sample_refs)
print(results_table["Lead-1"])


safe_run("BART-large", "facebook/bart-large-cnn")
safe_run("T5-small", "t5-small")
safe_run("FLAN-T5-base", "google/flan-t5-base")
safe_run("PEGASUS", "google/pegasus-cnn_dailymail")
safe_run("DistilBART", "sshleifer/distilbart-cnn-12-6")

# Heavy models
if torch.cuda.is_available():
    safe_run("FLAN-T5-large", "google/flan-t5-large")

try:
    safe_run("mBART", "facebook/mbart-large-cc25")
except:
    print("⚠️ mBART skipped")

try:
    safe_run("LED", "allenai/led-base-16384", max_input_tokens=1024)
except:
    print("⚠️ LED skipped")


# ============================================================
# 📊 FINAL TABLE
# ============================================================

import pandas as pd

results_df = pd.DataFrame(results_table).T
results_df = results_df.sort_values("ROUGE-1", ascending=False)

print("\n📊 FINAL BASELINE RESULTS\n")
print(results_df.to_string(float_format="%.4f"))

results_df.to_csv("baseline_results.csv")

print("\n✅ DONE — Results saved!")


[1] Lead-1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'ROUGE-1': 0.3175, 'ROUGE-2': 0.1402, 'ROUGE-L': 0.2688, 'BERTScore': 0.8691}

🚀 Running BART-large...


BART-large: 100%|██████████| 75/75 [03:07<00:00,  2.50s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ BART-large → {'ROUGE-1': 0.2801, 'ROUGE-2': 0.1149, 'ROUGE-L': 0.2321, 'BERTScore': 0.8633}

🚀 Running T5-small...


...

✅ FLAN-T5-large → {'ROUGE-1': 0.3894, 'ROUGE-2': 0.1845, 'ROUGE-L': 0.3353, 'BERTScore': 0.8805}

🚀 Running mBART...


mBART: 100%|██████████| 75/75 [05:19<00:00,  4.25s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ mBART → {'ROUGE-1': 0.0926, 'ROUGE-2': 0.0275, 'ROUGE-L': 0.0828, 'BERTScore': 0.7874}

🚀 Running LED...


LED:   0%|          | 0/75 [00:00<?, ?it/s]Input ids are automatically padded from 245 to 1024 to be a multiple of `config.attention_window`: 1024
Input ids are automatically padded from 125 to 1024 to be a multiple of `config.attention_window`: 1024
Input ids are automatically padded from 205 to 1024 to be a multiple of `config.attention_window`: 1024
Input ids are automatically padded from 100 to 1024 to be a multiple of `config.attention_window`: 1024
LED:   1%|▏         | 1/75 [00:03<04:36,  3.73s/it]Input ids are automatically padded from 72 to 1024 to be a multiple of `config.attention_window`: 1024
Input ids are automatically padded from 66 to 1024 to be a multiple of `config.attention_window`: 1024
Input ids are automatically padded from 71 to 1024 to be a multiple of `config.attention_window`: 1024
Input ids are automatically padded from 130 to 1024 to be a multiple of `config.attention_window`: 1024
LED:   3%|▎         | 2/75 [00:05<03:17,  2.70s/it]Input ids are automaticall

✅ LED → {'ROUGE-1': 0.2806, 'ROUGE-2': 0.1228, 'ROUGE-L': 0.2346, 'BERTScore': 0.8628}

📊 FINAL BASELINE RESULTS

               ROUGE-1  ROUGE-2  ROUGE-L  BERTScore
FLAN-T5-large   0.3894   0.1845   0.3353     0.8805
FLAN-T5-base    0.3822   0.1800   0.3328     0.8782
Lead-1          0.3175   0.1402   0.2688     0.8691
DistilBART      0.2852   0.1188   0.2351     0.8606
LED             0.2806   0.1228   0.2346     0.8628
BART-large      0.2801   0.1149   0.2321     0.8633
T5-small        0.2663   0.1074   0.2194     0.8562
PEGASUS         0.2391   0.0895   0.1924     0.8460
mBART           0.0926   0.0275   0.0828     0.7874

✅ DONE — Results saved!


In [16]:
# ============================================================
# CELL 5C: FINAL BASELINE RESULTS TABLE (NO CFMS)
# ============================================================

import pandas as pd

# ============================================================
# 🔥 CREATE DATAFRAME
# ============================================================

results_df = pd.DataFrame(results_table).T

# Ensure numeric values
results_df = results_df.apply(pd.to_numeric, errors="coerce")

# ============================================================
# 🔥 SORT BY PERFORMANCE
# ============================================================

results_df = results_df.sort_values("ROUGE-1", ascending=False)

# ============================================================
# 🔥 ADD RANK COLUMN
# ============================================================

results_df["Rank"] = results_df["ROUGE-1"].rank(
    ascending=False,
    method="min"
).astype(int)

# ============================================================
# 🔥 FORMAT (CLEAN PRINT)
# ============================================================

display_df = results_df.copy()

for col in ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BERTScore"]:
    display_df[col] = display_df[col].map(lambda x: f"{x:.4f}")

display_df["Rank"] = display_df["Rank"].astype(int)

# ============================================================
# 📊 PRINT TABLE
# ============================================================

print("\n" + "=" * 75)
print("        BASELINE EVALUATION RESULTS")
print("=" * 75)

print(display_df)

print("=" * 75)

# ============================================================
# 💾 SAVE FILE
# ============================================================

results_df.to_csv("baseline_results.csv")

print("\n📁 Saved as: baseline_results.csv")


        BASELINE EVALUATION RESULTS
              ROUGE-1 ROUGE-2 ROUGE-L BERTScore  Rank
FLAN-T5-large  0.3894  0.1845  0.3353    0.8805     1
FLAN-T5-base   0.3822  0.1800  0.3328    0.8782     2
Lead-1         0.3175  0.1402  0.2688    0.8691     3
DistilBART     0.2852  0.1188  0.2351    0.8606     4
LED            0.2806  0.1228  0.2346    0.8628     5
BART-large     0.2801  0.1149  0.2321    0.8633     6
T5-small       0.2663  0.1074  0.2194    0.8562     7
PEGASUS        0.2391  0.0895  0.1924    0.8460     8
mBART          0.0926  0.0275  0.0828    0.7874     9

📁 Saved as: baseline_results.csv
